# **SOLUTION**

In [ ]:
from array import array

class Solution(object):
    def lenOfVDiagonal(self, grid):
        n, m = len(grid), len(grid[0])
        nm = n * m

        # NE, SE, SW, NW
        dr = (-1, 1, 1, -1)
        dc = ( 1, 1,-1, -1)
        cw = (1, 2, 3, 0)

        # start2/start0 phẳng cho 4 hướng
        start2 = array('H', [0]) * (4 * nm)
        start0 = array('H', [0]) * (4 * nm)

        ans = 0
        g = grid  # alias

        # ---- helper: iterate ray starts per direction without tạo list ----
        def iterate_ray_starts(d):
            if d == 0:  # NE: starts: (i,0) and (n-1,j>=1)
                for i in range(n): yield i, 0
                for j in range(1, m): yield n - 1, j
            elif d == 1:  # SE: (0,j) and (i>=1,0)
                for j in range(m): yield 0, j
                for i in range(1, n): yield i, 0
            elif d == 2:  # SW: (0,j) and (i>=1,m-1)
                for j in range(m): yield 0, j
                for i in range(1, n): yield i, m - 1
            else:  # d == 3, NW: (i,m-1) and (n-1,j<=m-2)
                for i in range(n): yield i, m - 1
                for j in range(m - 1): yield n - 1, j

        # ---- helper: compute ray length L from (sr,sc) towards d (khỏi while/inb) ----
        def ray_len(d, sr, sc):
            if d == 0:   # NE: up-right
                up = sr + 1
                right = m - sc
                return up if up < right else right
            if d == 1:   # SE: down-right
                down = n - sr
                right = m - sc
                return down if down < right else right
            if d == 2:   # SW: down-left
                down = n - sr
                left = sc + 1
                return down if down < left else left
            else:        # NW: up-left
                up = sr + 1
                left = sc + 1
                return up if up < left else left

        # ==================== 1) build start2/start0 (duyệt NGƯỢC) ====================
        for d in range(4):
            rstep, cstep = dr[d], dc[d]
            base = d * nm
            for sr, sc in iterate_ray_starts(d):
                L = ray_len(d, sr, sc)
                # đi tới đuôi tia: r = sr + (L-1)*dr, c = sc + (L-1)*dc
                r = sr + (L - 1) * rstep
                c = sc + (L - 1) * cstep
                nxt_s2 = 0
                nxt_s0 = 0
                for _ in range(L):
                    v = g[r][c]
                    idx = base + r * m + c

                    # start2
                    if v == 2:
                        s2 = 1 + nxt_s0
                        start2[idx] = s2
                    else:
                        s2 = 0
                        start2[idx] = 0

                    # start0
                    if v == 0:
                        s0 = 1 + nxt_s2
                        start0[idx] = s0
                    else:
                        s0 = 0
                        start0[idx] = 0

                    if v == 1:
                        cand = 1 + nxt_s2
                        if cand > ans: ans = cand

                    # lùi lại
                    r -= rstep
                    c -= cstep
                    nxt_s2, nxt_s0 = s2, s0

        # ==================== 2) quét xuôi + ghép rẽ CW ====================
        for d in range(4):
            d2 = cw[d]
            rstep, cstep = dr[d], dc[d]
            rstep2, cstep2 = dr[d2], dc[d2]
            base2 = d2 * nm

            for sr, sc in iterate_ray_starts(d):
                L = ray_len(d, sr, sc)
                r = sr
                c = sc

                prev_has = False
                prev_val = 0
                prev_e2 = 0
                prev_e0 = 0

                for _ in range(L):
                    v = g[r][c]

                    # end1, end2, end0 tại (r,c)
                    e1 = 1 if v == 1 else 0

                    e2 = 0
                    if v == 2 and prev_has:
                        if prev_val == 1:
                            e2 = 2
                        elif prev_val == 0 and prev_e0 > 0:
                            e2 = prev_e0 + 1 if (prev_e0 + 1) > 2 else 2

                    e0 = (prev_e2 + 1) if (v == 0 and prev_has and prev_val == 2 and prev_e2 > 0) else 0

                    # ghép nhánh sau rẽ: ô ngay sau theo d2
                    qi = r + rstep2
                    qj = c + cstep2
                    if 0 <= qi < n and 0 <= qj < m:
                        jdx = base2 + qi * m + qj
                        s2 = start2[jdx]
                        s0 = start0[jdx]
                    else:
                        s2 = 0
                        s0 = 0

                    if e1:
                        cand = e1 + s2
                        if cand > ans: ans = cand
                    if e2:
                        cand = e2 + s0
                        if cand > ans: ans = cand
                    if e0:
                        cand = e0 + s2
                        if cand > ans: ans = cand

                    prev_has = True
                    prev_val = v
                    prev_e2 = e2
                    prev_e0 = e0

                    r += rstep
                    c += cstep

        return ans


# **DRAFT**

# **TESTCASES**

In [8]:
solution = Solution()

def run_test_case(case_number, input_grid, expected):
    result = solution.lenOfVDiagonal(input_grid)
    status = "PASSED" if result == expected else "FAILED"
    print(f"Case {case_number} - grid: {input_grid}, Output: {result}, Expected: {expected}, Status: {status}")

test_cases = [
    (1, [[2,2,1,2,2],[2,0,2,2,0],[2,0,1,1,0],[1,0,2,2,2],[2,0,0,2,2]], 5),
    (2, [[2,2,2,2,2],[2,0,2,2,0],[2,0,1,1,0],[1,0,2,2,2],[2,0,0,2,2]], 4),
    (3, [[1,2,2,2,2],[2,2,2,2,0],[2,0,0,0,0],[0,0,2,2,2],[2,0,0,2,0]], 5),
    (4, [[1]], 1)
]

for case_number, input_grid, expected in test_cases:
    run_test_case(case_number, input_grid, expected)

Case 1 - grid: [[2, 2, 1, 2, 2], [2, 0, 2, 2, 0], [2, 0, 1, 1, 0], [1, 0, 2, 2, 2], [2, 0, 0, 2, 2]], Output: 5, Expected: 5, Status: PASSED
Case 2 - grid: [[2, 2, 2, 2, 2], [2, 0, 2, 2, 0], [2, 0, 1, 1, 0], [1, 0, 2, 2, 2], [2, 0, 0, 2, 2]], Output: 4, Expected: 4, Status: PASSED
Case 3 - grid: [[1, 2, 2, 2, 2], [2, 2, 2, 2, 0], [2, 0, 0, 0, 0], [0, 0, 2, 2, 2], [2, 0, 0, 2, 0]], Output: 5, Expected: 5, Status: PASSED
Case 4 - grid: [[1]], Output: 1, Expected: 1, Status: PASSED
